Preferably, we automate the detection of disguised missing data

In [27]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
while project_root.name != "big_data_assignment" and project_root.parent != project_root:
    project_root = project_root.parent

project_root_str = str(project_root)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

print("Using project root:", project_root)
print("Ensured project root on sys.path")

Using project root: /Users/lisavanoosten/Desktop/Data Science/Big Data/Big-Data/big_data_assignment
Ensured project root on sys.path


In [28]:
from pathlib import Path
import duckdb

PROJECT_ROOT = project_root  # from the first cell
csv_path = PROJECT_ROOT / "data" / "raw" / "csv" / "train-1.csv"

con = duckdb.connect()
df = con.execute("SELECT * FROM read_csv_auto(?)", [str(csv_path)]).df()
df.head()

,column0,tconst,primaryTitle,originalTitle,startYear,endYear,runtimeMinutes,numVotes,label
0,4,tt0010600,The Doll,Die Puppe,1919,\N,66,1898.0,True
1,7,tt0011841,Way Down East,Way Down East,1920,\N,145,5376.0,True
2,9,tt0012494,Déstiny,Der müde Tod,1921,\N,97,5842.0,True
3,25,tt0015163,The Navigator,The Navigator,1924,\N,59,9652.0,True
4,38,tt0016220,The Phantom of the Opera,The Phantom of the Opera,1925,\N,93,17887.0,True


In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

def find_type_mismatch_tokens(series):
    """Return tokens that don't match the inferred dtype of the column."""
    s = series.astype(str).dropna()

    if pd.api.types.is_numeric_dtype(series):
        # treat anything that is NOT a valid number as suspicious
        mask = ~s.str.fullmatch(r"-?\d+(\.\d+)?")
        return s[mask].value_counts()

    # for non-numeric columns you can plug in other heuristics if you like
    return pd.Series(dtype=int)

# --- Shannon entropy of characters in a token ---
def char_entropy(token):
    if not isinstance(token, str) or len(token) == 0:
        return 0.0
    counts = Counter(token)
    probs = np.array(list(counts.values())) / len(token)
    return -np.sum(probs * np.log2(probs))

# --- Detect low-entropy tokens in a column ---
def detect_low_entropy_tokens(series,
                              min_freq_ratio=0.01,
                              max_entropy=1.0,
                              max_len=5):
    
    s = series.dropna().astype(str)
    total = len(s)
    
    freq = s.value_counts()
    freq_ratio = freq / total
    
    results = []
    for token, count in freq.items():
        ratio = freq_ratio[token]
        ent = char_entropy(token)
        
        if ratio >= min_freq_ratio and ent <= max_entropy and len(token) <= max_len:
            results.append({
                "token": token,
                "count": int(count),
                "freq_ratio": float(ratio),
                "entropy": float(ent),
                "length": len(token)
            })
    
    if not results:
        return pd.DataFrame(columns=["token", "count", "freq_ratio", "entropy", "length"])
    
    return pd.DataFrame(results).sort_values("freq_ratio", ascending=False)


# Run low-entropy detection for every column in the DataFrame
all_suspects = {}
for col in df.columns:
    print(f"\n=== Column: {col} ===")
    mismatches = find_type_mismatch_tokens(df[col])
    if not mismatches.empty:
        print("Type-mismatch candidates (possible disguised missing):")
        print(mismatches.head(20))

    col_suspects = detect_low_entropy_tokens(df[col])
    if not col_suspects.empty:
        print("Low-entropy tokens:")
        print(col_suspects.head(20))


=== Column: column0 ===

=== Column: tconst ===

=== Column: primaryTitle ===

=== Column: originalTitle ===

=== Column: startYear ===
Low-entropy tokens:
  token  count  freq_ratio   entropy  length
0    \N     94    0.097612  1.000000       2
1  2000     20    0.020768  0.811278       4
2  2020     16    0.016615  1.000000       4
3  2002     16    0.016615  1.000000       4
4  1991     10    0.010384  1.000000       4

=== Column: endYear ===
Low-entropy tokens:
  token  count  freq_ratio  entropy  length
0    \N    869    0.902388      1.0       2

=== Column: runtimeMinutes ===
Low-entropy tokens:
   token  count  freq_ratio   entropy  length
0     90     38    0.039460  1.000000       2
1     93     34    0.035306  1.000000       2
2    100     33    0.034268  0.918296       3
3     95     32    0.033229  1.000000       2
4     92     31    0.032191  1.000000       2
5     88     31    0.032191 -0.000000       2
6     87     25    0.025961  1.000000       2
7     91     23    0